# Multivariable Optimization

Companion notebook for the [Multivariable Optimization](https://ml-viz-ruby.vercel.app/courses/calculus-for-ml/03-multivariable-optimization) lesson.

We work the same example as the lesson, $f(x, y) = x^2 + xy + y^2 - 3x$, end to end:

1. Find the critical point by solving $\nabla f = 0$.
2. Build the Hessian and verify it with finite differences.
3. Classify the critical point from the Hessian's eigenvalues.
4. Take one **Newton step** and watch it land exactly on the minimum, then compare to a gradient step.
5. Run the eigenvalue convexity test, and study how the learning rate relates to $\lambda_{\max}$.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## Intuition — optimizing over many variables

With one variable you set `f'(x) = 0`. With many, you set the **gradient** to zero:
`∇f = 0` marks a *critical point*. But a flat gradient alone doesn't say whether you're
at a valley, a peak, or a saddle — for that you need **curvature**, captured by the
**Hessian** (the matrix of second partials). Its eigenvalues classify the point, its
inverse powers **Newton's method** (jump straight to the minimum of a quadratic), and
its largest eigenvalue caps how big a gradient-descent step can be before it diverges.
This notebook works one quadratic end-to-end through all of that.

## 1. From scratch — critical point, Hessian, classification

## The worked example: $f(x,y) = x^2 + xy + y^2 - 3x$

We define the function, its analytic gradient, and its analytic Hessian.

$$\nabla f = \begin{bmatrix} 2x + y - 3 \\ x + 2y \end{bmatrix},
\qquad
\mathbf{H} = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}.$$

The Hessian is constant because $f$ is quadratic.

In [ ]:
def f(p):
    x, y = p
    return x**2 + x*y + y**2 - 3*x

def grad_f(p):
    x, y = p
    return np.array([2*x + y - 3, x + 2*y])

def hessian_f(p):
    # Constant Hessian for this quadratic; argument kept for a general API.
    return np.array([[2., 1.],
                     [1., 2.]])

# --- Step 1: solve grad_f = 0 for the critical point ---
# 2x + y - 3 = 0 and x + 2y = 0  ->  H @ [x, y] = [3, 0]
A = np.array([[2., 1.],
              [1., 2.]])
b = np.array([3., 0.])
critical = np.linalg.solve(A, b)
print("Critical point (solved grad_f = 0): {}".format(critical))
print("grad_f at critical point:          {}".format(grad_f(critical)))
print("f at critical point:               {:.4f}".format(f(critical)))


**What to notice:** solving the linear system `H·[x,y] = [3,0]` gives the critical
point `(2, −1)`, and the gradient there is `[0, 0]` — confirmed. Because `f` is quadratic
its Hessian is *constant*, which is what makes the next two steps exact.

### Verify the Hessian with finite differences

The analytic Hessian above should match a numerical estimate. We approximate each
second partial with the central-difference stencil

$$H_{ij} \approx \frac{f(\mathbf{x}+h e_i + h e_j) - f(\mathbf{x}+h e_i - h e_j) - f(\mathbf{x}-h e_i + h e_j) + f(\mathbf{x}-h e_i - h e_j)}{4h^2}.$$

In [ ]:
def numerical_hessian(f, p, h=1e-4):
    """Central-difference estimate of the Hessian via the 4-point stencil above."""
    n = len(p)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            pp = p.copy(); pp[i] += h; pp[j] += h
            pm = p.copy(); pm[i] += h; pm[j] -= h
            mp = p.copy(); mp[i] -= h; mp[j] += h
            mm = p.copy(); mm[i] -= h; mm[j] -= h
            H[i, j] = (f(pp) - f(pm) - f(mp) + f(mm)) / (4 * h * h)
    return H

H_numeric = numerical_hessian(f, critical)
print('numerical Hessian:\n', H_numeric.round(4))
print('analytic  Hessian:\n', hessian_f(critical))
assert np.allclose(H_numeric, hessian_f(critical), atol=1e-3), "FD Hessian must match analytic"
print('\nfinite-difference Hessian matches the analytic one ✓')

**What to notice:** the finite-difference Hessian reproduces `[[2,1],[1,2]]` — the
analytic second partials are correct. This 4-point stencil is the 2-D analogue of the
central difference from Lesson 1, now catching bugs in *second* derivatives.

With a verified Hessian we can **classify** the critical point from its eigenvalue
signs — and repeat the test on the canonical minimum / saddle / maximum trio.

In [ ]:
def classify_critical_point(H):
    eigs = np.linalg.eigvalsh(H)  # eigvalsh: H is symmetric, eigenvalues are real
    if np.all(eigs > 0):  return 'Local minimum (all lambda > 0)'
    if np.all(eigs < 0):  return 'Local maximum (all lambda < 0)'
    if np.all(eigs == 0): return 'Degenerate'
    return 'Saddle point (mixed lambda signs)'

# Our worked example first, then the canonical min / saddle / max trio.
hessians = {
    'f=x^2+xy+y^2-3x  at (2,-1)': hessian_f(critical),  # the lesson example
    'f=x^2+y^2        at (0,0)':  np.array([[2., 0.], [0., 2.]]),
    'f=x^2-y^2        at (0,0)':  np.array([[2., 0.], [0., -2.]]),
    'f=-x^2-y^2       at (0,0)':  np.array([[-2., 0.], [0., -2.]]),
}

for name, H in hessians.items():
    eigs = np.linalg.eigvalsh(H)
    print(name)
    print('  Eigenvalues:    {}'.format(np.round(eigs, 4)))
    print('  Classification: {}\n'.format(classify_critical_point(H)))


**What to notice:** the worked example has eigenvalues `1` and `3` — both positive, so
it's a **minimum**. The trio confirms the rule: all-positive → bowl, mixed signs →
saddle, all-negative → dome. The eigenvalues *are* the curvatures along the Hessian's
principal axes.

## 2. The library way — optimizer + autodiff Hessian

In practice you hand the function to an optimizer and let autodiff supply derivatives.
`scipy.optimize.minimize` finds the minimum numerically, and `jax.hessian` builds the
whole second-derivative matrix by differentiating twice — no hand calculus, no finite
differences. The cell checks both against our by-hand results.

In [ ]:
from scipy.optimize import minimize
import jax, jax.numpy as jnp

res = minimize(f, x0=np.array([0.0, 0.0]))          # numerical optimizer
print('scipy minimum :', res.x.round(4), ' f =', round(float(res.fun), 4))
assert np.allclose(res.x, critical, atol=1e-4), "optimizer must find (2, -1)"

def f_jax(p):
    return p[0]**2 + p[0]*p[1] + p[1]**2 - 3*p[0]

H_auto = np.array(jax.hessian(f_jax)(jnp.array(critical)))
print('jax.hessian   :\n', H_auto)
assert np.allclose(H_auto, hessian_f(critical)), "autodiff Hessian must match analytic"
print('\nscipy minimum and autodiff Hessian both match the by-hand work ✓')

**What to notice:** `scipy.optimize.minimize` lands on `(2, −1)` and `jax.hessian`
returns exactly `[[2,1],[1,2]]`. The by-hand derivation was a teaching tool; in real
code you'd call these and trust autodiff for the curvature.

## One Newton step vs. one gradient step

Newton's method minimizes the second-order Taylor model exactly:

$$\mathbf{s} = -\mathbf{H}^{-1}\nabla f(\mathbf{x}_0), \qquad \mathbf{x} \leftarrow \mathbf{x}_0 + \mathbf{s}.$$

Starting from $\mathbf{x}_0 = (0,0)$ we have $\nabla f = (-3, 0)$, and the single Newton
step should land exactly on the minimum $(2, -1)$ because $f$ is quadratic. A plain
gradient step only crawls along the gradient direction.

In [ ]:
x0 = np.array([0.0, 0.0])
g0 = grad_f(x0)
H0 = hessian_f(x0)

# Newton step: solve H s = -g  (equivalent to s = -inv(H) @ g, but solve is stabler)
newton_step = np.linalg.solve(H0, -g0)
x_newton = x0 + newton_step

# One gradient step at a sensible learning rate (eta = 1 / lambda_max).
eta = 1.0 / np.max(np.linalg.eigvalsh(H0))
x_grad = x0 - eta * g0

print("Start x0:          {}".format(x0))
print("grad at x0:        {}".format(g0))
print("Newton step s:     {}".format(newton_step))
print("After Newton step: {}   f = {:.4f}".format(x_newton, f(x_newton)))
print("After grad step:   {}   f = {:.4f}   (eta = {:.3f})".format(x_grad, f(x_grad), eta))
print("True minimum:      {}   f = {:.4f}".format(critical, f(critical)))
print("\nNewton landed on the minimum exactly? {}".format(
    np.allclose(x_newton, critical)))

# Now iterate gradient descent to show how many steps it needs to catch up.
w = x0.copy()
for k in range(1, 1001):
    w = w - eta * grad_f(w)
    if np.linalg.norm(w - critical) < 1e-6:
        print("Gradient descent reached the minimum after {} steps.".format(k))
        break


**What to notice:** the single **Newton step** lands *exactly* on `(2, −1)` — for a
quadratic, minimizing the second-order Taylor model is minimizing `f` itself. The plain
gradient step only inches along `[−3,0]` and needs many iterations to catch up. Newton
buys that speed by paying for the Hessian (and its inverse) every step.

## The eigenvalue convexity test

A quadratic $f(\mathbf{x}) = \tfrac12 \mathbf{x}^\top \mathbf{H}\mathbf{x}$ is convex iff
$\mathbf{H} \succeq 0$, i.e. every eigenvalue is $\geq 0$. We reproduce the two
matrices worked by hand in the lesson.

In [ ]:
def is_convex_quadratic(H):
    """A quadratic x^T H x is convex iff H is positive semi-definite."""
    eigenvalues = np.linalg.eigvalsh(H)  # eigvalsh for symmetric H
    return bool(np.all(eigenvalues >= 0))

H_convex = np.array([[2., 1.], [1., 3.]])   # hand-derived eigenvalues (5 +/- sqrt 5)/2
H_saddle = np.array([[1., 2.], [2., 1.]])   # hand-derived eigenvalues -1 and 3

for label, H in [("H_convex", H_convex), ("H_saddle", H_saddle)]:
    eigs = np.linalg.eigvalsh(H)
    print("{}: eigenvalues = {}  ->  convex = {}".format(
        label, np.round(eigs, 4), is_convex_quadratic(H)))


**What to notice:** `H_convex` has all-positive eigenvalues → convex bowl; `H_saddle`
has a negative one → not convex. Convexity is exactly Hessian positive-semidefiniteness,
and it's what guarantees gradient descent reaches the *global* minimum.

### The shapes of loss landscapes

The Hessian's signature shows up as the *shape* of the surface. Below are the three
archetypes: a positive-definite **bowl** (one minimum), an indefinite **saddle**, and a
**non-convex** ripple with many minima.

In [ ]:
x = np.linspace(-2, 2, 100)
y = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(x, y)

functions = {
    'Local minimum\n(H positive definite)': X**2 + Y**2,
    'Saddle point\n(H indefinite)':          X**2 - Y**2,
    'Non-convex\n(multiple minima)':         np.sin(3*X) * np.cos(3*Y),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), subplot_kw={'projection': '3d'})

for ax, (title, Z) in zip(axes, functions.items()):
    surf = ax.plot_surface(X, Y, Z, cmap='twilight', alpha=0.85,
                           linewidth=0, antialiased=True)
    ax.set_title(title, pad=10, fontsize=10)
    ax.set_xlabel('w₁'); ax.set_ylabel('w₂'); ax.set_zlabel('Loss')
    ax.tick_params(labelsize=7)

plt.suptitle('Loss Landscape Shapes', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

**What to notice:** the bowl curves up everywhere (descent always works), the saddle
curves up one way and down the other (a zero gradient that *isn't* a minimum), and the
rippled surface has many basins — the reality for deep networks, where the optimizer can
only promise a *local* minimum.

### Learning rate vs curvature

For a quadratic with Hessian eigenvalues up to `λ_max = L`, gradient descent is stable
only for `η < 2/L`, fastest near `η = 1/L`, and **diverges** above `2/L`. The sweep below
uses `f = 2w₁² + 0.5w₂²` (so `L = 4`, threshold `η = 0.5`) at four rates straddling it.

In [ ]:
# Quadratic f(w) = 2 w1^2 + 0.5 w2^2, so H = diag(4, 1): lambda_max = 4, lambda_min = 1.
def loss(w): return 2*w[0]**2 + 0.5*w[1]**2
def grad(w): return np.array([4*w[0], w[1]])

H_lr = np.array([[4., 0.], [0., 1.]])
lam_max = np.max(np.linalg.eigvalsh(H_lr))
eta_opt = 1.0 / lam_max          # fastest stable, balanced step
eta_diverge = 2.0 / lam_max      # iteration diverges above this threshold
print("lambda_max = {:.1f}  ->  eta* = 1/L = {:.3f},  diverges when eta > 2/L = {:.3f}".format(
    lam_max, eta_opt, eta_diverge))

learning_rates = [0.05, 0.2, 0.49, 0.51]  # last two straddle the 0.5 threshold
w0 = np.array([2.0, 2.0])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-2.5, 2.5, 200))
Z = 2*xx**2 + 0.5*yy**2

for ax, lr in zip(axes, learning_rates):
    path = [w0.copy()]
    w = w0.copy()
    for _ in range(40):
        w = w - lr * grad(w)
        path.append(w.copy())
        if np.any(np.abs(w) > 100):
            break
    path = np.array(path)

    ax.contourf(xx, yy, Z, levels=15, cmap='twilight', alpha=0.6)
    ax.contour(xx, yy, Z, levels=15, colors='white', alpha=0.2, linewidths=0.5)
    ax.plot(path[:, 0], path[:, 1], 'o-', color='#f97316', ms=4, lw=1.5)
    ax.scatter(*path[0], color='#2dd4bf', s=80, zorder=5, label='Start')
    ax.scatter(0, 0, marker='*', color='#f59e0b', s=150, zorder=5, label='Min')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal'); ax.grid(False)
    final_loss = loss(path[-1])
    status = 'DIVERGED' if final_loss > 100 else 'Loss={:.3f}'.format(final_loss)
    ax.set_title('eta = {}\n{}'.format(lr, status), fontsize=10)

plt.suptitle('Effect of Learning Rate (optimal eta = 1/L = 0.25, diverges for eta > 0.5)',
             y=1.04, fontsize=12)
plt.tight_layout(); plt.show()

**What to notice:** `η = 0.05` and `0.2` converge (0.2 faster), `0.49` zig-zags but
still creeps in, and `0.51` — just past `2/L = 0.5` — **diverges**, each step overshooting
further. The safe step size is set by the *sharpest* direction (`λ_max`), even though most
directions are flatter — the root cause of slow training on ill-conditioned problems.

## 4. Gotchas & failure modes

- **A zero gradient isn't a minimum.** Saddle points also have `∇f = 0`; you must check
  the Hessian. High-dimensional loss surfaces are riddled with saddles.
- **Learning rate has a hard ceiling.** `η ≥ 2/λ_max` diverges — no amount of patience
  helps. The ceiling is set by the sharpest curvature direction.
- **Ill-conditioning slows descent.** A large `λ_max/λ_min` ratio forces a tiny step for
  the sharp direction while the flat direction crawls — GD zig-zags. (Momentum/Adam and
  preconditioning target this.)
- **Newton needs a well-behaved Hessian.** If `H` is singular or indefinite, `−H⁻¹∇f`
  can point *uphill* or blow up — pure Newton is unsafe far from a minimum.

In [ ]:
# A saddle: gradient is zero but it's not a minimum
Hs = np.array([[2., 0.], [0., -1.]])
print('saddle eigenvalues:', np.linalg.eigvalsh(Hs), '-> zero gradient, yet not a min')

# Conditioning: step is capped by lambda_max, convergence paced by lambda_min
for cond in [1, 10, 100]:
    H = np.array([[cond, 0.], [0., 1.]])
    lo, hi = np.linalg.eigvalsh(H)[[0, -1]]
    print(f'cond={cond:>3}:  eta_max = 2/lambda_max = {2/hi:.3f},  '
          f'slow direction shrinks by only {1 - (1/hi)*lo:.3f} per step')

**What to notice:** the saddle has a positive *and* a negative eigenvalue — a critical
point that traps naive first-order methods. And as the condition number grows, the safe
step `2/λ_max` shrinks while the flat direction barely moves per step: a `cond=100`
problem needs ~100× more iterations. This is *why* plain gradient descent is slow and why
adaptive optimizers exist.

## Key takeaways

- **Critical points** solve `∇f = 0`; the **Hessian's eigenvalues** classify them —
  all-positive = minimum, mixed = saddle, all-negative = maximum.
- **Newton's method** (`x − H⁻¹∇f`) minimizes a quadratic in one step but needs the
  Hessian; **gradient descent** is cheaper but paced by curvature.
- **Convex ⇔ Hessian ⪰ 0**, which guarantees a global minimum.
- Learning rate is capped at `2/λ_max`; **ill-conditioning** (`λ_max/λ_min`) is the
  hidden reason first-order methods crawl.
- In practice: `scipy.optimize.minimize` + `jax.hessian` instead of hand calculus.

**Next:** [Jacobians](https://ml-viz-ruby.vercel.app/courses/calculus-for-ml/04-jacobians)
— the gradient generalized to vector-valued functions.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The gradient of the worked example

Differentiate the lesson's function $f(x, y) = x^2 + xy + y^2 - 3x$ partial by partial:

$$\nabla f = \begin{bmatrix} \partial f / \partial x \\ \partial f / \partial y \end{bmatrix}
= \begin{bmatrix} 2x + y - 3 \\ x + 2y \end{bmatrix}$$

The checks verify the critical point $(2, -1)$ from the lesson and cross-check against finite differences at several other points.

In [ ]:
def grad_f(x, y):
    """Gradient of f(x, y) = x^2 + x*y + y^2 - 3x as a length-2 array."""
    # TODO(you): partial wrt x: 2x + y - 3
    dfdx = ...

    # TODO(you): partial wrt y: x + 2y
    dfdy = ...

    return np.array([dfdx, dfdy])

In [ ]:
# Checks — run me
assert np.allclose(grad_f(2, -1), [0, 0]), "(2, -1) is the critical point from the lesson"

f = lambda x, y: x ** 2 + x * y + y ** 2 - 3 * x
h = 1e-6
for (px, py) in [(0.0, 0.0), (1.0, 2.0), (-3.0, 0.5)]:
    nx = (f(px + h, py) - f(px - h, py)) / (2 * h)
    ny = (f(px, py + h) - f(px, py - h)) / (2 * h)
    assert np.allclose(grad_f(px, py), [nx, ny], atol=1e-5), f"gradient mismatch at {(px, py)}"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def grad_f(x, y):
    dfdx = 2 * x + y - 3
    dfdy = x + 2 * y
    return np.array([dfdx, dfdy])
```

</details>

### Exercise 2 — Classify critical points by curvature

At a critical point the gradient is zero — the **Hessian's eigenvalues** decide what kind of point it is:

- all $\lambda_i > 0$ → curves up in every direction → **minimum**
- all $\lambda_i < 0$ → curves down in every direction → **maximum**
- mixed signs → up one way, down another → **saddle**

Implement the test with `np.linalg.eigvalsh` (the symmetric-matrix eigensolver). The last check is the classic trap: a Hessian with *positive diagonal* can still be a saddle once the off-diagonals get big.

In [ ]:
def classify_critical_point(H):
    """Return 'minimum', 'maximum', or 'saddle' from the Hessian H (symmetric)."""
    # TODO(you): eigenvalues of H (hint: np.linalg.eigvalsh)
    lams = ...

    # TODO(you): all positive -> minimum; all negative -> maximum; otherwise saddle
    ...

In [ ]:
# Checks — run me
assert classify_critical_point([[2, 1], [1, 2]]) == "minimum", "eigenvalues 1, 3 -> bowl"
assert classify_critical_point([[-2, 0], [0, -3]]) == "maximum", "eigenvalues -2, -3 -> dome"
assert classify_critical_point([[2, 0], [0, -1]]) == "saddle", "mixed signs -> saddle"
assert classify_critical_point([[1, 2], [2, 1]]) == "saddle", "positive diagonal yet eigenvalues -1, 3"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def classify_critical_point(H):
    lams = np.linalg.eigvalsh(np.asarray(H, dtype=float))
    if np.all(lams > 0):
        return "minimum"
    if np.all(lams < 0):
        return "maximum"
    return "saddle"
```

</details>